# EconoCausal: Customer Segmentation and Campaign Strategy
*Author: Principal Machine Learning Architect & Senior Causal AI Researcher*  
*Target Environment: Production / Research-Grade ML Pipeline*

---

## 1. Executive Summary & Role of Segmentation in Causal Marketing

Following the rigorous identification, estimation, and refutation of causal effects in Notebooks 01–06, we now possess robust, customer-level point estimates of the **Conditional Average Treatment Effect (CATE)** for both conversion ($\hat{	au}_{conv}(X)$) and spend ($\hat{	au}_{spend}(X)$). However, raw continuous predictions of uplift are insufficient for direct operational execution. 

This notebook serves as the vital translation layer between econometric modeling and enterprise marketing strategy. The primary objective is to map continuous individual treatment effects (ITE) into an actionable, discretely bounded **Customer Segmentation Framework**.

### The Causal Uplift Marketing Paradigm
Standard predictive marketing models score customers based on their absolute likelihood to buy ($P(Y=1 | X)$), often targeting those with the highest propensity. This naive approach suffers from treatment inefficiency by targeting individuals who would have purchased regardless of the campaign (wasted spend) and ignoring customers whose behavior could have been changed.

Causal uplift modeling shifts the objective function from predicting the outcome to predicting the *incremental change* in the outcome caused by the treatment. This enables the division of the customer base into the **4-Quadrant Uplift Framework**:

1. **Persuadables** (High Uplift): Customers who respond positively to the campaign. *Targeting Priority: High.*
2. **Sure Things** (Low Uplift, High Baseline): Customers who purchase anyway. *Targeting Priority: Low (Suppress to save budget).*
3. **Lost Causes** (Low Uplift, Low Baseline): Customers who will not purchase regardless. *Targeting Priority: Low.*
4. **Sleeping Dogs** (Negative Uplift): Customers who react negatively to the campaign (e.g., unsubscribing due to annoyance). *Targeting Priority: Strict Exclusion.*

### Notebook Scope & Constraints
This notebook strictly focuses on **analytical and diagnostic segmentation**. It produces structured segments, thresholds, profiles, and strategy taxonomies. It **does not** perform final budget optimization, fractional knapsack allocations, or automated deployment logic—these prescriptive constraints are reserved for Notebook 08.


In [ ]:
# 2. Environment Setup & Dependency Configuration
import os
import sys
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Suppress minor warnings for publication-quality output
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# Ensure Numba JIT doesn't cause Windows App Control issues during plotting if any underlying libs use it
os.environ["NUMBA_DISABLE_JIT"] = "1"

# Configure publication-grade plotting
plt.style.use('default')
sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'legend.fontsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight'
})

# Define Project Paths
PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"
PLOTS_DIR = MODELS_DIR / "segment_plots"

PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# Deterministic reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger(__name__)

logger.info("Environment initialized and paths configured.")


In [ ]:
# 3. Loading & Inspecting Segmentation Artifacts

def load_table(path_stem: Path) -> pd.DataFrame:
    """Load table with parquet priority and csv fallback."""
    parquet_p = path_stem.with_suffix(".parquet")
    csv_p = path_stem.with_suffix(".csv")
    if parquet_p.exists():
        # engine="fastparquet" avoids pyarrow Period bugs on windows
        return pd.read_parquet(parquet_p, engine="fastparquet")
    elif csv_p.exists():
        return pd.read_csv(csv_p)
    else:
        raise FileNotFoundError(f"Neither {parquet_p} nor {csv_p} exists.")

logger.info("Loading upstream causal artifacts...")

# 1. Dataset & Metadata
df_processed = load_table(DATA_DIR / "processed" / "processed_data")
with open(MODELS_DIR / "feature_metadata.json", "r") as f:
    feature_metadata = json.load(f)

# 2. Uplift Rankings & CATE Predictions
# We load the main rankings and CATE summary artifacts from Notebook 06
uplift_rankings = load_table(MODELS_DIR / "uplift_rankings")
cate_conv = load_table(MODELS_DIR / "cate_predictions_conversion")
cate_spend = load_table(MODELS_DIR / "cate_predictions_spend")

# We expect baseline models or outcomes. The outcome variables are 'visit', 'conversion', 'spend'
T_COL = feature_metadata["treatment"]

# 3. Validation
assert len(uplift_rankings) == len(df_processed), "Mismatch in row counts between dataset and rankings."
logger.info(f"Successfully loaded {len(df_processed):,} customer records.")

# Display Artifact Inventory
print("=" * 60)
print("ARTIFACT INVENTORY & DIMENSIONALITY")
print("=" * 60)
print(f"Base Dataset:        {df_processed.shape}")
print(f"CATE Conversion:     {cate_conv.shape}")
print(f"CATE Spend:          {cate_spend.shape}")
print(f"Uplift Rankings:     {uplift_rankings.shape}")
print(f"Treatment Variable:  {T_COL}")
print(f"Confounders:         {len(feature_metadata.get('confounders', []))} features")
print("=" * 60)


## 4. Segmentation Framework Specification

The segmentation objective is to map continuous individual-level treatment effects (ITE) into the **4-Quadrant Framework**. This requires two dimensions of information per customer:
1. **Predicted Causal Uplift ($\hat{	au}(X)$):** The expected incremental response due to treatment.
2. **Predicted Baseline Response ($P(Y=1 | T=0, X)$):** The organic probability of the outcome in the absence of treatment.

### Threshold Definition
To operationalize this, we must define threshold parameters:
- **Uplift Threshold ($\gamma$)**: Delineates high vs. low incremental response. For conversion, we typically look for a statistically and practically significant positive effect (e.g., top deciles or absolute $> 0.005$).
- **Baseline Threshold ($lpha$)**: Delineates high vs. low organic likelihood of outcome.

### The 4 Quadrants
| Quadrant | Baseline ($\ge lpha$) | Uplift ($\ge \gamma$) | Strategic Implication |
| :--- | :---: | :---: | :--- |
| **Persuadables** | Low/High | **High** | Primary target. Campaign changes behavior. |
| **Sure Things** | **High** | Low/Neg | Suppress. They buy anyway. |
| **Lost Causes** | Low | Low/Neg | Ignore. Baseline low and campaign doesn't help. |
| **Sleeping Dogs** | Low/High | **Negative** | Active harm. Strict suppression. |

*Note: In some marketing domains, Sleeping Dogs are defined strictly by highly negative uplift, whereas Lost Causes hover around zero. We will separate Negative Uplift from Zero/Low Uplift.*


In [ ]:
# 5 & 6. Construction of Segment Scores & Threshold Design

# For the baseline response, we can use the original dataset's control group averages or proxy with the actual target
# However, to be completely rigorous, we need an estimation of Y(0). 
# Since we didn't export baseline predictions from the DML explicitly, we can proxy baseline response 
# for standard segmentation by looking at historical interaction features like 'history' or 'recency', 
# or by splitting the customer base at the median of uplift.

# Let's define the thresholds for Conversion
# Persuadables: Uplift > 0.005 (0.5% absolute lift)
# Sleeping Dogs: Uplift < -0.001 (Negative lift)
# Sure Things vs Lost Causes depends on baseline propensity. 
# We'll construct a heuristic baseline proxy using historical spend ('history') as a proxy for organic loyalty.

def calculate_segmentation_thresholds(uplift_series, is_spend=False):
    """Calculates data-driven thresholds for causal uplift."""
    # Define Sleeping dogs as bottom 5% or negative, Persuadables as top 20% or substantial positive
    if is_spend:
        sleeping_dog_thresh = min(-0.1, uplift_series.quantile(0.05))
        persuadable_thresh = max(0.5, uplift_series.quantile(0.80))
    else:
        sleeping_dog_thresh = min(-0.001, uplift_series.quantile(0.05))
        persuadable_thresh = max(0.005, uplift_series.quantile(0.80))
        
    return persuadable_thresh, sleeping_dog_thresh

# Calculate Thresholds
conv_persuadable_th, conv_sleeping_dog_th = calculate_segmentation_thresholds(uplift_rankings['CATE_Conversion_Mens'])
spend_persuadable_th, spend_sleeping_dog_th = calculate_segmentation_thresholds(uplift_rankings['CATE_Spend_Mens'], is_spend=True)

# Define a High Baseline Proxy threshold (e.g., top 30% of historical spend)
high_baseline_thresh = df_processed['history'].quantile(0.70) if 'history' in df_processed.columns else 0

thresholds = {
    "conversion": {
        "persuadable_gamma": float(conv_persuadable_th),
        "sleeping_dog_gamma": float(conv_sleeping_dog_th)
    },
    "spend": {
        "persuadable_gamma": float(spend_persuadable_th),
        "sleeping_dog_gamma": float(spend_sleeping_dog_th)
    },
    "baseline_alpha_proxy": float(high_baseline_thresh)
}

print("=" * 50)
print("COMPUTED SEGMENTATION THRESHOLDS")
print("=" * 50)
print(f"Conversion Persuadable (Gamma):  >= {conv_persuadable_th:.4f}")
print(f"Conversion Sleeping Dog:         <= {conv_sleeping_dog_th:.4f}")
print(f"Spend Persuadable (Gamma):       >= {spend_persuadable_th:.4f}")
print(f"Spend Sleeping Dog:              <= {spend_sleeping_dog_th:.4f}")
print(f"High Baseline Proxy (Alpha):     >= {high_baseline_thresh:.2f}")
print("=" * 50)


In [ ]:
# 7. Customer Segmentation Implementation

def assign_quadrants(df, uplift_col, th_persuadable, th_sleeping, base_proxy_col, th_baseline):
    """Assigns each row to a causal quadrant."""
    conditions = [
        (df[uplift_col] >= th_persuadable),
        (df[uplift_col] <= th_sleeping),
        (df[uplift_col] > th_sleeping) & (df[uplift_col] < th_persuadable) & (df[base_proxy_col] >= th_baseline),
        (df[uplift_col] > th_sleeping) & (df[uplift_col] < th_persuadable) & (df[base_proxy_col] < th_baseline)
    ]
    choices = ['Persuadables', 'Sleeping Dogs', 'Sure Things', 'Lost Causes']
    
    # Default to Lost Causes if logic escapes
    return np.select(conditions, choices, default='Lost Causes')

# Create a segmentation master table
seg_df = df_processed.copy()

# For simplicity in strategy, we focus on the "Mens E-Mail" intervention as our primary treatment variant, 
# or max uplift over variants. We'll use Mens E-Mail CATE for this baseline segmentation.
seg_df['CATE_Conversion'] = uplift_rankings['CATE_Conversion_Mens'].values
seg_df['CATE_Spend'] = uplift_rankings['CATE_Spend_Mens'].values

# Assign Segments
seg_df['Segment_Conversion'] = assign_quadrants(
    seg_df, 'CATE_Conversion', conv_persuadable_th, conv_sleeping_dog_th, 'history', high_baseline_thresh
)

seg_df['Segment_Spend'] = assign_quadrants(
    seg_df, 'CATE_Spend', spend_persuadable_th, spend_sleeping_dog_th, 'history', high_baseline_thresh
)

# Convert to Categorical for ordered plotting
quadrant_order = ['Persuadables', 'Sure Things', 'Lost Causes', 'Sleeping Dogs']
seg_df['Segment_Conversion'] = pd.Categorical(seg_df['Segment_Conversion'], categories=quadrant_order, ordered=True)
seg_df['Segment_Spend'] = pd.Categorical(seg_df['Segment_Spend'], categories=quadrant_order, ordered=True)

logger.info("Segmentation assignment complete.")


In [ ]:
# 8. Segment Distribution & Population Analysis

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Conversion Segments
conv_counts = seg_df['Segment_Conversion'].value_counts(normalize=True).sort_index() * 100
sns.barplot(x=conv_counts.index, y=conv_counts.values, ax=axes[0], palette=['#2ca02c', '#1f77b4', '#7f7f7f', '#d62728'])
axes[0].set_title('Conversion Uplift: Customer Segments', fontweight='bold')
axes[0].set_ylabel('Percentage of Customer Base (%)')
axes[0].set_xlabel('')
for p in axes[0].patches:
    axes[0].annotate(f"{p.get_height():.1f}%", (p.get_x() + p.get_width() / 2., p.get_height()), 
                     ha='center', va='center', xytext=(0, 5), textcoords='offset points')

# Spend Segments
spend_counts = seg_df['Segment_Spend'].value_counts(normalize=True).sort_index() * 100
sns.barplot(x=spend_counts.index, y=spend_counts.values, ax=axes[1], palette=['#2ca02c', '#1f77b4', '#7f7f7f', '#d62728'])
axes[1].set_title('Spend Uplift: Customer Segments', fontweight='bold')
axes[1].set_ylabel('Percentage of Customer Base (%)')
axes[1].set_xlabel('')
for p in axes[1].patches:
    axes[1].annotate(f"{p.get_height():.1f}%", (p.get_x() + p.get_width() / 2., p.get_height()), 
                     ha='center', va='center', xytext=(0, 5), textcoords='offset points')

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'fig1_segment_distribution.png')
plt.show()

# Print Summary Table
print("CONVERSION SEGMENT SUMMARY")
display(seg_df.groupby('Segment_Conversion', observed=True)['CATE_Conversion'].agg(['count', 'mean', 'median', 'std', 'min', 'max']).round(4))

print("\nSPEND SEGMENT SUMMARY")
display(seg_df.groupby('Segment_Spend', observed=True)['CATE_Spend'].agg(['count', 'mean', 'median', 'std', 'min', 'max']).round(4))


In [ ]:
# 9. Segment Profiling by Customer Attributes

# We analyze how key pre-treatment covariates map to segments to generate actionable personas.
profile_cols = ['history', 'recency', 'mens', 'womens', 'newbie']

profile_summary = seg_df.groupby('Segment_Conversion', observed=True)[profile_cols].mean()

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(profile_summary.T, cmap='YlGnBu', annot=True, fmt=".2f", linewidths=.5, ax=ax)
ax.set_title("Customer Attribute Profiles by Conversion Segment", fontweight='bold')
plt.xlabel("Causal Segment")
plt.ylabel("Covariate Averages")
plt.savefig(PLOTS_DIR / 'fig2_segment_feature_heatmap.png')
plt.show()

logger.info("Notice how 'Persuadables' might differ fundamentally in historical spend or recency compared to 'Lost Causes'.")


## 10. Campaign Strategy Interpretation

The strategic translation of the causal segments directly feeds into the upcoming policy engine:

1. **Persuadables:** These customers exhibit high sensitivity to the campaign. They represent the primary target audience. The campaign strategy here is **Maximal Allocation** up to the budget constraint.
2. **Sure Things:** While they have high loyalty (baseline response), their incremental uplift is negligible. The campaign strategy is **Suppression/Hold-Out** to save ROI, as they cannibalize profit if subsidized.
3. **Lost Causes:** They neither respond organically nor incrementably. The campaign strategy is **Exclusion**, though cheap or distinct alternative interventions could be tested in future A/B tests.
4. **Sleeping Dogs:** The campaign actively discourages their conversion or spend. The campaign strategy is **Strict Do Not Contact (DNC)**.

## 11. Outcome-Specific Segment Analysis
A customer might be a "Persuadable" for Conversion (they click and visit) but a "Lost Cause" for Spend (they buy low-margin/cheap items, generating no incremental revenue).


In [ ]:
# Visualizing Conversion vs Spend Segment Overlap
cross_tab = pd.crosstab(seg_df['Segment_Conversion'], seg_df['Segment_Spend'], normalize='index') * 100

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cross_tab, annot=True, fmt=".1f", cmap="Blues", cbar_kws={'label': '% Overlap'})
ax.set_title("Segment Transition: Conversion vs Spend", fontweight='bold')
plt.xlabel("Spend Segment")
plt.ylabel("Conversion Segment")
plt.savefig(PLOTS_DIR / 'fig3_conversion_vs_spend_crosstab.png')
plt.show()


In [ ]:
# 12. Validation and Sanity Checks

# Ensure that Persuadables have strictly positive average uplift
conv_persuadable_mean = seg_df[seg_df['Segment_Conversion'] == 'Persuadables']['CATE_Conversion'].mean()
assert conv_persuadable_mean > 0, "Validation Failed: Persuadables have non-positive mean uplift."

# Ensure that Sleeping Dogs have negative uplift
conv_sleeping_mean = seg_df[seg_df['Segment_Conversion'] == 'Sleeping Dogs']['CATE_Conversion'].mean()
if not pd.isna(conv_sleeping_mean):
    assert conv_sleeping_mean < 0, "Validation Failed: Sleeping Dogs have non-negative mean uplift."

logger.info(f"Sanity Checks Passed. Persuadable Mean Uplift: {conv_persuadable_mean:.4f} | Sleeping Dog Mean Uplift: {conv_sleeping_mean:.4f}")


In [ ]:
# 14. Artifact Persistence & Reproducibility

# Export Segments
export_df = seg_df[['CATE_Conversion', 'CATE_Spend', 'Segment_Conversion', 'Segment_Spend']].copy()
# Save as parquet utilizing fastparquet for pyarrow bypass
export_df.to_parquet(MODELS_DIR / "customer_segments.parquet", index=False, engine="fastparquet")

# Export Thresholds
with open(MODELS_DIR / "segment_thresholds.json", "w") as f:
    json.dump(thresholds, f, indent=4)

# Export Profiles
profile_summary.to_csv(MODELS_DIR / "segment_profiles.csv")

logger.info("Segmentation artifacts successfully persisted to models/ directory.")

# Print final manifest
print("=" * 60)
print("NOTEBOOK 07 EXPORTED ARTIFACT MANIFEST")
print("=" * 60)
print("  [SUCCESS] models/customer_segments.parquet")
print("  [SUCCESS] models/segment_thresholds.json")
print("  [SUCCESS] models/segment_profiles.csv")
print("  [SUCCESS] models/segment_plots/fig1_segment_distribution.png")
print("  [SUCCESS] models/segment_plots/fig2_segment_feature_heatmap.png")
print("  [SUCCESS] models/segment_plots/fig3_conversion_vs_spend_crosstab.png")
print("=" * 60)


## 15. Transition to Notebook 08: Budget Optimization

We have successfully mapped the continuous point estimates of causal treatment effects into a rigid, operationally meaningful customer segmentation framework. We now know *who* responds incrementally, *who* buys anyway, and *who* reacts negatively.

In **Notebook 08 (Budget Optimization & Recommendation)**, we will utilize these segment assignments alongside the raw `uplift_rankings` to build a mathematically constrained allocation policy. We will simulate realistic marketing budget constraints, use mixed-integer/fractional knapsack heuristics to select the optimal subset of Persuadables to target, and estimate the final expected Return on Investment (ROI) of the causal policy versus baseline random targeting.
